# Instruments Category Analysis (Supabase)

This notebook loads `public.instruments` and analyzes `category` counts split by `asset_type` (`ETF` vs `Stock`).

In [ ]:
import os
from pathlib import Path
import pandas as pd
from supabase import create_client


In [ ]:
def load_env(path: Path):
    if not path.exists():
        return
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip()
        if k and k not in os.environ:
            os.environ[k] = v

root = Path.cwd()
load_env(root / '.env.local')
load_env(root / '.env')

SUPABASE_URL = os.getenv('NEXT_PUBLIC_SUPABASE_URL') or os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_SERVICE_ROLE_KEY') or os.getenv('SUPABASE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError('Missing Supabase env vars: NEXT_PUBLIC_SUPABASE_URL/SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY/SUPABASE_KEY')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('Supabase ready')

In [ ]:
# Pull instruments table in pages
rows = []
page_size = 1000
offset = 0

while True:
    resp = (
        supabase
        .table('instruments')
        .select('symbol,name,asset_type,market,sector,industry,category,valuation')
        .range(offset, offset + page_size - 1)
        .execute()
    )
    batch = resp.data or []
    if not batch:
        break
    rows.extend(batch)
    offset += len(batch)

df = pd.DataFrame(rows)
print(f'Loaded rows: {len(df):,}')
df.head()

In [ ]:
# Normalize asset_type/category fields
df['asset_type_norm'] = df['asset_type'].fillna('').astype(str).str.strip().str.upper()
df['category_norm'] = df['category'].fillna('').astype(str).str.strip()

summary = (
    df.groupby('asset_type_norm', dropna=False)
      .agg(
          total=('symbol', 'count'),
          category_blank=('category_norm', lambda s: (s == '').sum()),
          category_non_blank=('category_norm', lambda s: (s != '').sum())
      )
      .sort_values('total', ascending=False)
)
summary['blank_pct'] = (summary['category_blank'] / summary['total'] * 100).round(2)
summary

In [ ]:
# Top category counts for ETF and Stock
def top_categories(asset_type: str, top_n: int = 50):
    sub = df[df['asset_type_norm'] == asset_type].copy()
    sub = sub[sub['category_norm'] != '']
    out = (
        sub.groupby('category_norm', dropna=False)
           .agg(count=('symbol', 'count'))
           .sort_values('count', ascending=False)
           .head(top_n)
    )
    out['share_pct'] = (out['count'] / out['count'].sum() * 100).round(2)
    return out

etf_top = top_categories('ETF', top_n=100)
stock_top = top_categories('Stock', top_n=100)

print('ETF top categories')
display(etf_top.head(30))
print('Stock top categories')
display(stock_top.head(30))

In [ ]:
# Optional: export full counts for review
out_dir = Path('notebooks')
out_dir.mkdir(parents=True, exist_ok=True)

(
    df[df['category_norm'] != '']
      .groupby(['asset_type_norm', 'category_norm'], dropna=False)
      .size()
      .reset_index(name='count')
      .sort_values(['asset_type_norm', 'count'], ascending=[True, False])
      .to_csv(out_dir / 'category_counts_by_asset_type.csv', index=False)
)
print('Wrote notebooks/category_counts_by_asset_type.csv')